<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12_power_grid/Ex12.1_pinn_grid_state_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 — The State of a Grid

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L12.1 · Power Grid Stability Estimation**

A transmission system operator must know the voltage, magnitude and phase
angle, at every bus of the grid: this is the **state** of the grid. Only some
of these quantities are measured, and every measurement has an error. **State
estimation** computes the state from the measurements and the power flow
equations. You do it in two ways - weighted least squares, the method used in
control centres, and a neural network that returns the state and is trained
with the power flow equations - and compare the two on accuracy and computing
time. In the second part a short circuit on a line makes the rotor angle and
speed of the local generator oscillate, and you estimate the generator's
inertia constant from the frequency recorded at its terminals.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and weighted least squares on three sets of meters for an agreed 0.01 p.u. | — |
| **5** | a physics-informed neural network for the state, and the comparison | the power flow, the loss |
| **6** | the inverse problem: the inertia of a machine from a disturbance | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About two minutes on a CPU. Numbers are printed with at most two
decimals, except errors smaller than that.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem_12_1.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12_power_grid/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem_12_1"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem_12_1.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem_12_1 as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex12.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

The grid has six buses (nodes) connected by six lines. Bus 0 is the
connection to the Nordic transmission system, bus 1 a local power plant,
buses 2 to 4 three load areas, and bus 5 an HVDC connection. The **state** of
the grid is the voltage at every bus, given by its magnitude $V_i$ and its
phase angle $\theta_i$. The angle at bus 0 is the reference and is set to
zero, so the state consists of eleven values: six magnitudes and five angles.
The task is to compute these eleven values from a small number of
measurements that contain noise.

## 2 · The data

| | |
|---|---|
| grid | 6 buses, 6 lines, per unit on a 100 MVA base |
| operating points | the power generated or consumed at every bus scaled by a random factor between 0.6 and 1.2 |
| a phasor measurement unit (PMU) | measures the voltage magnitude $V$ (error 0.004 p.u.) and phase angle $\theta$ (error 0.002 rad) at its bus |
| a power meter (SCADA) | measures the active and reactive power injected at its bus, $P$ and $Q$, i.e. generation minus load (error 0.010 p.u.) |
| a forecast | an estimate of $P$ and $Q$ at a bus without a power meter (error 0.05 p.u., assumed) |

Three sets of meters are compared:

| set | what is measured | number of measurements |
|---|---|---|
| `full` | PMUs at buses 0 and 3, power meters at buses 0 to 4 | 14 |
| `thin` | a PMU at bus 0, power meters at buses 0 and 3 | 6 |
| `forecast` | the thin set, plus a forecast of $P$ and $Q$ at buses 1, 2, 4 and 5 | 14 |

**The grid is representative of eastern Denmark, not a model of it.** Its
line impedances are textbook values, not measured ones: no operator publishes
those. The accuracy of a forecast is assumed.

## 3 · The physics

**The power flow equations** (L12.1's *The Power Flow Equations*). At every
bus $i$ the power injected equals the power that leaves along the lines:

$$P_i = V_i \sum_k V_k \big(G_{ik}\cos\theta_{ik} + B_{ik}\sin\theta_{ik}\big), \qquad
  Q_i = V_i \sum_k V_k \big(G_{ik}\sin\theta_{ik} - B_{ik}\cos\theta_{ik}\big)$$

with $\theta_{ik} = \theta_i - \theta_k$, and $G$ and $B$ the real and
imaginary parts of the admittance matrix: what the lines are. They are
algebraic. Nothing moves in them.

**Every measurement is a function of the state.** A PMU measures $V_i$ and
$\theta_i$ directly. A power meter measures the active power $P_i$ or the
reactive power $Q_i$ injected at its bus - the power generated there minus
the power consumed - and the power flow equations above give both from the
state. This function, from the state to the measurements, is called $h$.

The cell below draws the grid with the meters of each set, prints the
injected power and the voltage at every bus at nominal load, and counts how
many of the eleven state values each set of meters can determine.

In [ ]:
pb.draw_meters(); plt.show()                                 # the grid and the three sets of meters
pb.describe_problem()
Y = pb.build_ybus()                                           # the admittance matrix of the six lines

**What you should see.** Voltages between 0.971 and 1.000 p.u. and angles
within 2.5°, the city (bus 3) lowest. The full set and the forecast set
determine all 11 unknowns. **The thin set determines 5 of them**: six
measurements cannot determine eleven values, however accurate the meters (L12.1's *Fewer
Measurements Than Unknowns*).

---

## 4 · The reference, and weighted least squares

**Two classical methods with two different roles.** This section uses two
classical methods, and only one of them is an estimator.

| | Newton–Raphson power flow | weighted least squares (WLS) |
|---|---|---|
| role here | builds the **reference**: the true states the estimators are scored against | the **classical state estimator**, the one the network is compared with |
| starts from | the exact injected power $P$ and $Q$ at every bus | the measurements, with their errors, at some buses |
| equations and unknowns | as many equations as unknowns | usually more measurements than unknowns, which disagree slightly |
| returns | the one state that satisfies all equations | the state that comes closest to all measurements |
| used in practice for | planning: what the state will be for a given generation and load | operation: what the state is now, from the meters |

Newton–Raphson cannot estimate the state of a real grid, because the exact
injected power at every bus is exactly what is not known in operation. It is
used here only because the exercise needs true states to score the
estimators against, and in a computed example they can be made.

**The reference** is the power flow calculation. For each of 300 operating
points the injected powers are given, the **Newton–Raphson method** solves the
power flow equations to $10^{-10}$, and the state it returns is taken as the
true state. The measurements are then computed from this state, with noise
added.

**Why an iterative solver, and not an exact solution.** The power flow
equations of section 3 are nonlinear - the unknowns appear as products
$V_iV_k$ and inside $\cos\theta_{ik}$ and $\sin\theta_{ik}$ - and for a grid
with more than two buses and closed loops of lines they have no solution in
closed form. Newton–Raphson starts from a guess (all magnitudes 1 p.u., all
angles zero), replaces the equations by their linear approximation at the
guess, solves that linear system for a correction, and repeats; near the
solution the error is squared at every iteration, so four or five iterations
reach $10^{-10}$. It is iterative, not a discretisation: nothing is divided
into steps in space or time.

**The classical estimator** is weighted least squares: the state $x$ that
minimises

$$J(x) = \sum_j \Big(\frac{z_j - h_j(x)}{\sigma_j}\Big)^2$$

where $z_j$ is measurement $j$, $h_j(x)$ what that meter would measure if the
state were $x$, and $\sigma_j$ the meter's standard error (L12.1's *Weighted
Least Squares, and What It Costs*).

**Why it is not the Newton–Raphson power flow.** State estimation starts from
measurements that contain errors, and usually from more of them than there
are unknowns (14 for 11 in the full set). No state reproduces all of them
exactly, because they contradict each other slightly, so weighted least
squares looks for the state that comes **closest** to all of them, each
difference counted in units of its meter's error - an accurate meter weighs
more than a forecast. It is solved by **Gauss–Newton** steps: linearise at
the current guess, correct, repeat, each step a least-squares problem. With
fewer independent measurements than unknowns, as in the thin set, no method
can determine the state.

**The agreed accuracy is 0.01 p.u.**: the worst bus within 1 % of nominal
voltage, on average over the operating points. The cell below runs the
estimator on each of the three sets of meters (`pb.wls_all`).

In [ ]:
V_true, th_true = pb.scenarios(300, seed=7)                   # 300 operating points: the reference states
AGREED = 0.01

def score(V, th):
    # the worst bus of each operating point, averaged: magnitude in p.u., angle in degrees
    return float(np.abs(V - V_true).max(axis=1).mean()), float(np.degrees(np.abs(th - th_true).max(axis=1).mean()))

wls, est = {}, {}
for kind in ("full", "thin", "forecast"):
    ms_k = pb.meters(kind)
    Z_k = pb.readings(V_true, th_true, ms_k, seed=3)          # what the meters read, with noise
    t0 = time.perf_counter(); V_k, th_k = pb.wls_all(Z_k, ms_k); sec = (time.perf_counter() - t0) / len(Z_k)
    wls[kind] = score(V_k, th_k) + (sec,)
    est[kind] = (V_k, th_k)                                   # the estimates themselves, for section 5's comparison
    print(f"  weighted least squares, {kind:9s}: worst bus {wls[kind][0]:.4f} p.u. and {wls[kind][1]:.2f} deg, {1000 * sec:.2f} ms an estimate"
          + ("   <- meets 0.01" if wls[kind][0] <= AGREED else ""))

**What you should see.** With the full set the worst bus is within 0.003 p.u.
and 0.07°. **With the thin set it is off by 0.025 p.u. and 1.8°**, which is
more than the whole spread of voltages on this grid: the estimator returns
numbers, and at the buses with no meter they are close to a guess. With the
forecasts added it is within 0.005 p.u. and 0.15° again. An estimate takes
a few milliseconds.

---

## 5 · A physics-informed neural network (PINN) for the state of the grid

**The network returns the state, and the power flow equations measure its
error** (L12.1's *The Network Learns the State, Not the Physics*). The network
takes the $M$ measurements of a set of meters as input and returns the state:
for a grid of $B$ buses, $B$ voltage magnitudes and $B - 1$ angles (one angle
is the reference), $2B - 1$ values. Here $B = 6$, so the state has 11 values,
and the network uses the forecast set, $M = 14$. For another grid or another
set of meters only $B$ and $M$ change.

The network is not trained on true states, which are never known in
practice. Training proceeds in three steps, repeated:

1. the network returns a state for each set of measurements;
2. the power flow equations compute, from that state, the value each meter
   would show if the state were correct;
3. these computed values are compared with the values the meters actually
   showed, and the weights of the network are adjusted to make the
   difference smaller.

When the difference is as small as the meters' own errors allow, the state
the network returns is consistent with both the measurements and the
physics.

**A PINN of a different kind.** The network is physics-informed in the sense
of L7: its loss is the physics, here the power flow equations, and it is never
shown a true state. It differs from the PINNs of L7 to L10 - and from the one
of section 6 below - in three ways:

| | PINN of L7 to L10, and of section 6 | the network of this section |
|---|---|---|
| input | coordinates: position and time | the measurements |
| output | the solution at that point, e.g. a temperature | the whole state of the grid: $2B - 1$ values |
| physics in the loss | a differential equation, its derivatives taken by automatic differentiation at collocation points | algebraic equations, no derivatives with respect to the input |
| what one training gives | the solution of one problem | an estimator for any set of measurements, trained over 3000 operating points |

It learns a map from measurements to state, with the physics as the training
signal, rather than solving one equation over space and time.

**Why a dense network.** The input is always the same $M$ measurements, in
the same order, from the same grid, and the output always the same $2B - 1$
values; the state changes smoothly as the measurements change. Learning a
smooth map from one vector of fixed length to another is what a dense
network is for: no spatial grid for a convolution, no sequence in time for a
recurrent network, no changing topology for a graph network. Its speed comes
from replacing an iterative solve by one pass through the network. The price
is that it has learned this one grid: the lines are fixed into its weights,
it has no input through which a line outage could be reported, and it must
be trained again when the grid changes. A **graph network**, whose weights
are shared by all buses, would be the choice when the lines change or the
grid is large (Ex_12.2 uses one); a **recurrent network** when the state is
tracked in time from a stream of measurements rather than estimated from one
snapshot.

**TODO 1 — the power flow in torch.** The two sums of section 3, for every
state of a batch at once; `h` then reads off what each of the 14 meters
should show in that state.

In [ ]:
ms = pb.meters("forecast")                                    # the 14 readings the network is given
G, B = to_tensor(Y.real), to_tensor(Y.imag)                   # the admittance matrix, real and imaginary parts
SIGMA = to_tensor(ms.sigma).reshape(-1)                       # the accuracy of each reading

def power(V, th):
    # the power flow equations for a batch of states: V and th of shape (N, 6) -> P and Q of shape (N, 6)
    dth = th[:, :, None] - th[:, None, :]                     # the angle differences theta_i - theta_k
    VV = V[:, :, None] * V[:, None, :]                        # the products V_i V_k
    P = (VV * (G * torch.cos(dth) + B * torch.sin(dth))).sum(dim=2)   # active power injected at each bus
    Q = (VV * (G * torch.sin(dth) - B * torch.cos(dth))).sum(dim=2)   # reactive power injected at each bus
    return P, Q

def h(V, th):
    # what each meter of the set should read in the state (V, th): shape (N, 14)
    P, Q = power(V, th)
    return torch.stack([{"V": V, "TH": th, "P": P, "Q": Q}[kind][:, bus] for kind, bus in ms.spec], dim=1)

**The network, and the two things built in** (L12.1's *Build In What You Know
for Certain*):

| | |
|---|---|
| type | dense feed-forward network (multilayer perceptron): every neuron of a layer connected to every neuron of the next |
| input | the $M$ measurements (here 14), each scaled to zero mean and unit standard deviation over the training set |
| hidden layers | 3, of 64 neurons each, activation $\tanh$ |
| output layer | $2B - 1$ values (here 11), linear, then the two constraints below |
| weights | 9 995, Xavier initialisation, random seed 88 |
| training data | 3000 operating points, their measurements only (no true state) |

The reference angle is not an output: it is zero. The magnitudes pass through
a logistic function, so they lie between 0.90 and 1.10 p.u. whatever the
weights are; the five angles are the outputs multiplied by 0.3 rad, the order
of the angles on this grid, so the outputs themselves stay of order one.

In [ ]:
class Estimator(torch.nn.Module):                             # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=len(ms), n_out=2 * pb.N_BUS - 1, n_hidden=64, n_layers=3)   # dense tanh network: 14 readings in, 11 numbers out
    def forward(self, z):
        out = self.net(z)                                     # the network's 11 raw outputs
        V = 0.90 + 0.20 * torch.sigmoid(out[:, :pb.N_BUS])    # six magnitudes, between 0.90 and 1.10 p.u. by construction
        th = torch.cat([torch.zeros_like(out[:, :1]), 0.3 * out[:, pb.N_BUS:]], dim=1)   # the reference angle is zero; five more
        return V, th

V_train, th_train = pb.scenarios(3000, seed=1)                # 3000 operating points to train on
Z_train = pb.readings(V_train, th_train, ms, seed=5)          # their readings. The states are not used again.
mean, spread = Z_train.mean(axis=0), Z_train.std(axis=0) + 1e-6
scaled = lambda Z: to_tensor((Z - mean) / spread).to(DEVICE)  # the readings as the network takes them: zero mean, unit spread

**TODO 2 — the loss. It needs no true state.** It is the same $J$ as in
section 4, with the network's output in place of $x$, divided by the number
of measurements $M$ and averaged over the $K = 3000$ operating points of the
training set:

$$\mathcal{L} = \frac{1}{K}\sum_{k=1}^{K}\ \frac{1}{M}\sum_{j=1}^{M} \Big(\frac{z_j^{(k)} - h_j\big(\mathcal{N}(z^{(k)})\big)}{\sigma_j}\Big)^2$$

with $z^{(k)}$ the measurements of operating point $k$ and $\mathcal{N}(z^{(k)})$
the state the network returns for them. Weighted least squares minimises $J$
again, by several Gauss–Newton steps, for every new set of measurements; the
network is trained once to return the minimiser for any set, so that
afterwards one estimate costs a single evaluation (L12.1's *What a Graph
Network Adds, and What It Does Not*).

In [ ]:
model = Estimator().to(DEVICE)                                # the network, on the device
Z_in, Z_obs = scaled(Z_train), to_tensor(Z_train).to(DEVICE)  # the readings, scaled for the network and as read

def loss_fn():
    V, th = model(Z_in)                                       # the state the network proposes for each set of readings
    return (((Z_obs - h(V, th)) / SIGMA) ** 2).mean()         # what the meters should read against what they read

Training is the Part 2 recipe - Adam, then L-BFGS - with the loss of TODO 2:
1500 Adam steps at learning rate $3\times10^{-3}$, then 30 L-BFGS steps, the
full training set at every step. About half a minute.

In [ ]:
print(f"{parameter_count(model)} weights, {len(Z_train)} sets of readings, no true state")
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=1500, lbfgs_steps=30, lr=3e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

The comparison: the network against weighted least squares on the 300
held-out operating points of section 4 - accuracy, the time of one estimate,
and of 300 at once - and then a line out that nobody reported, where the
network's own check $J$ is what remains when no truth is at hand.

In [ ]:
Z_test = pb.readings(V_true, th_true, ms, seed=3)             # the 300 held-out operating points of section 4
def estimate(Z):
    with torch.no_grad():  # no gradient tracking: evaluation only
        V, th = model(scaled(Z))                              # the network's state for each set of measurements
        J = (((to_tensor(Z).to(DEVICE) - h(V, th)) / SIGMA) ** 2).mean(dim=1)   # its own check: how well the state explains the measurements
    return to_numpy(V), to_numpy(th), to_numpy(J)
V_net, th_net, J_net = estimate(Z_test)
net_score = score(V_net, th_net)
t_one = middle(lambda: estimate(Z_test[:1]))                  # one estimate at a time
t_batch = middle(lambda: estimate(Z_test)) / len(Z_test)      # 300 at once
V_wls, th_wls = est["forecast"]

def rmse(V, th):
    # root-mean-square error over all buses and operating points: magnitude in p.u., angle in degrees
    return float(np.sqrt(((V - V_true) ** 2).mean())), float(np.degrees(np.sqrt(((th - th_true) ** 2).mean())))
rmse_wls, rmse_net = rmse(V_wls, th_wls), rmse(V_net, th_net)

print(f"{'forecast set, 300 operating points':38s}{'RMSE |V|':>12s}{'RMSE angle':>12s}{'worst bus |V|':>15s}{'one estimate':>14s}{'training':>10s}")
print(f"  {'weighted least squares':36s}{rmse_wls[0]:8.4f} p.u.{rmse_wls[1]:8.3f} deg{wls['forecast'][0]:11.4f} p.u.{1000 * wls['forecast'][2]:11.2f} ms{'-':>10s}")
print(f"  {'network, trained with the power flow':36s}{rmse_net[0]:8.4f} p.u.{rmse_net[1]:8.3f} deg{net_score[0]:11.4f} p.u.{1000 * t_one:11.2f} ms{train_time:8.0f} s")
print(f"  the network on all 300 at once: {1e6 * t_batch:.1f} microseconds an estimate")

# a line the estimators are not told about: line 1-2 is out
Y_out = pb.build_ybus(outage=1)
V_out, th_out = pb.scenarios(100, seed=9, Y=Y_out)            # 100 operating points of the changed grid
Z_out = pb.readings(V_out, th_out, ms, seed=9, Y=Y_out)
worst = lambda V: float(np.abs(V - V_out).max(axis=1).mean())
V_a, _ = pb.wls_all(Z_out, ms, Y_out); V_b, _ = pb.wls_all(Z_out, ms, Y); V_c, _, J_out = estimate(Z_out)
print(f"line 1-2 out: least squares told of it {worst(V_a):.4f} p.u., not told {worst(V_b):.4f} p.u., the network {worst(V_c):.4f} p.u.")
print(f"the network's own check J: median {np.median(J_net):.2f} on the grid it was trained on, {np.median(J_out):.2f} with the line out")

bus = np.arange(pb.N_BUS); wdt = 0.38
per_bus = lambda e: np.sqrt((e ** 2).mean(axis=0))            # RMSE of each bus over the 300 operating points
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].bar(bus - wdt / 2, per_bus(V_wls - V_true), wdt, label="weighted least squares")
ax[0].bar(bus + wdt / 2, per_bus(V_net - V_true), wdt, label="network")
ax[0].axhline(AGREED, color="0.4", ls="--", lw=1); ax[0].text(-0.4, AGREED, "agreed accuracy 0.01", va="bottom", fontsize=8)
ax[0].set_ylim(0, 1.15 * AGREED); ax[0].set_xlabel("bus"); ax[0].set_ylabel("RMSE of |V|  [p.u.]")
ax[0].set_title("voltage magnitude: lower is better, 0 is perfect", fontsize=9); ax[0].legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(0, 0.84))
ax[1].bar(bus - wdt / 2, np.degrees(per_bus(th_wls - th_true)), wdt, label="weighted least squares")
ax[1].bar(bus + wdt / 2, np.degrees(per_bus(th_net - th_true)), wdt, label="network")
ax[1].set_xlabel("bus"); ax[1].set_ylabel("RMSE of the angle  [deg]")
ax[1].set_title("voltage angle (bus 0 is the reference, so 0)", fontsize=9)
ax[2].boxplot([J_net, J_out], showfliers=False)
ax[2].set_xticks([1, 2], ["the grid it was\ntrained on", "line 1-2 out,\nnobody told"])
ax[2].axhline(3 / 14, color="0.4", ls="--", lw=1); ax[2].text(1.55, 3 / 14, "expected with the\nright grid model, 3/14", va="bottom", fontsize=8)
ax[2].set_yscale("log"); ax[2].set_ylabel("the network's own check J")
ax[2].set_title("how well its state explains the measurements", fontsize=9)
plt.tight_layout(); plt.savefig(cc.output_path("Ex12.1_report.png"), dpi=150); plt.show()

**How to read the results.** The table gives two measures of the error for
each estimator. The **root-mean-square error (RMSE)** is taken over all buses
and all 300 operating points: the typical size of the error. The **worst
bus** is the largest error of each operating point, averaged: the measure the
agreed accuracy of 0.01 p.u. refers to. Ideally both would be zero; they
cannot be, because the meters have errors, and the best an estimator can do
from these meters is what weighted least squares achieves.

**What you should see.** The RMSE of the voltage magnitude is about 0.004
p.u. for both estimators, the network's slightly lower, and the RMSE of the
angle about 0.09° for both; the worst bus is within about 0.005 p.u.: **the
same accuracy as weighted least squares**, from the same measurements, without
ever having been shown a true state. One estimate takes a few tenths of a
millisecond, and 300 at once a few microseconds each, against a few
milliseconds for least squares. Training took about half a minute.

**The figure.** The left and middle panels show the RMSE **at each bus**, the
two estimators side by side. All magnitude errors are about 0.004 p.u. - the
error of one PMU reading - and far below the agreed 0.01 p.u. (dashed). The
angle error grows with the distance from bus 0, whose angle is the reference
and therefore exact, and is largest at bus 5, where only a forecast is
available. Neither estimator is better at any bus.

**The right panel** is the network's own check, $J$ for each operating point
(the box holds the middle half of the operating points, the line in it is the
median). It measures how well the state the network returns explains the
measurements, in units of the meters' errors. With the right model of the
grid it cannot be zero: $M$ noisy measurements are fitted with $2B - 1$
values, and the expected value of $J$ is then $(M - 2B + 1)/M$, here
$(14 - 11)/14 = 3/14 \approx 0.21$ (dashed);
the network's median, about 0.23, is at that value. **With a line out and
nobody told**, both estimators are wrong by about 0.05 p.u., because the
model of the grid both use is wrong; least squares told of the outage is
right again, while the network would have to be trained again. But the check
rises about six times, to a median of about 1.3 (the scale is logarithmic):
the state no longer explains the measurements, and that can be seen with no
true state at hand (L12.1's *How You Know the Estimate Is Any Good*).

---

## 6 · The inverse problem: the inertia of a machine

**Why estimate the inertia.** The aim is not to detect the disturbance: the
short circuit is detected and cleared by the protection, and here it is
known. The aim is to **use** the disturbance to measure a property of the
grid that no meter shows: the **inertia** of a generator, the kinetic energy
stored in its rotating masses. Inertia slows the change of frequency when
generation and load suddenly differ, and so gives the frequency control time
to act. As converter-connected wind and solar plants replace synchronous
generators, the inertia of a grid falls and changes from hour to hour, and
the operator needs to know it to decide how much fast reserve must be held.
The inertia of a unit determines how its frequency moves after a
disturbance, so it can be estimated from a recording of that frequency.

**The disturbance** (L12.1's *The Swing Equation*). A short circuit occurs on
line 0–1 at 0.5 s, and the protection disconnects the line at 0.6 s. The
rotor of the generator at the local plant first accelerates and then
decelerates relative to the rest of the grid: its rotor angle oscillates, by
up to 28°. The rotor angle $\delta$ and speed $\omega$ obey

$$\frac{d\delta}{dt} = \omega - \omega_s, \qquad
  \frac{2H}{\omega_s}\frac{d\omega}{dt} = P_m - P_e(\delta) - D\,\frac{\omega - \omega_s}{\omega_s}$$

with $P_m$ the mechanical power, $P_e(\delta)$ the electrical power it
delivers into the grid, $D$ the damping, $\omega_s$ the synchronous speed and
$H$ the **inertia constant** in seconds: the stored kinetic energy over the
rating. $H$ is the number wanted. It is 4.0 s in the record, which is
synthetic: a PMU at the plant, 50 readings of frequency a second, noise of
0.002 Hz.

In [ ]:
rec = pb.disturbance()                                        # the record: t, f, and the constants of the one-machine model
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axvspan(0.0, 0.45, color="0.6", alpha=0.20, label="window before the fault (0 to 0.45 s)")
ax.axvspan(rec["t_fault"], rec["t_clear"], color="tab:red", alpha=0.45, label=f"short circuit on line 0-1, {rec['t_fault']:.1f} to {rec['t_clear']:.1f} s")
ax.axvspan(rec["t_clear"], rec["t_clear"] + 2.0, color="tab:green", alpha=0.12, label="window after the fault (0.6 to 2.6 s)")
ax.plot(rec["t"], rec["f"], lw=1.2, color="tab:blue", label="frequency measured by the PMU at the plant")
ax.axhline(50.0, color="0.5", lw=0.8, ls="--")
ax.set_xlabel("time  [s]"); ax.set_ylabel("frequency  [Hz]"); ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
ax.set_title("the recorded disturbance: the plant speeds up during the short circuit, then oscillates", fontsize=9)
plt.tight_layout(); plt.show()
print(f"  frequency between {rec['f'].min():.2f} and {rec['f'].max():.2f} Hz; largest angle swing of the plant {rec['separation']:.0f} deg")

**What you should see.** Before the fault the frequency is 50 Hz within the
meter noise: nothing moves. During the short circuit the plant cannot deliver
its power to the grid, its rotor speeds up and its frequency rises, here by
0.7 Hz in 0.1 s; after the line is opened the frequency oscillates about 50 Hz,
about 2.6 times a second, and the oscillation decays only slowly because the
damping is small. Both the rate of rise during the fault and the rate of the
oscillation depend on the inertia - a heavier rotor rises more slowly and
oscillates more slowly - and that is the information the fits below use.

**The model is the plant against a fixed grid.** In the record the grid is
not quite fixed (its own inertia is 150 s, not infinite), so the model is a
little wrong, as models are.

**Two ways again.** The classical one integrates the equation from the start
of the window for a trial $H$ and adjusts it by least squares, `pb.shoot_inertia`.
The physics-informed one trains a network for $\delta(t)$ and $\omega(t)$
**and** the number $H = e^{\beta}$ together: the loss is the residual of the
two equations plus the misfit to the recorded frequency. Each is started from
an inertia twice too large and from one twice too small, on the two seconds
after the fault is cleared - and then on the 0.45 s before the fault, when
nothing is happening.

In [ ]:
WS = rec["ws"]                                                # the synchronous speed, 2 pi 50 rad/s

class Swing(torch.nn.Module):                                 # a network for the swing, and one more trainable number
    def __init__(self, h_start):
        super().__init__()
        self.net = MLP(n_in=1, n_out=2, n_hidden=48, n_layers=4)    # dense tanh network: time in, angle and speed out
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(h_start))))   # H = exp(beta), trained beside the weights
    def forward(self, tau):
        out = self.net(tau)
        return out[:, :1], 5.0 * out[:, 1:]                   # the angle in rad, and the speed above synchronous in rad/s

def identify(lo, hi, h_start):
    # H from the record between lo and hi seconds, by the physics-informed fit started at h_start
    set_seed(88)                                              # fix the random start so a run repeats
    keep = (rec["t"] >= lo - 1e-9) & (rec["t"] <= hi + 1e-9)
    tau = to_tensor(((rec["t"][keep] - lo) / (hi - lo)).reshape(-1, 1), requires_grad=True).to(DEVICE)   # scaled time, differentiable
    f_obs = to_tensor(rec["f"][keep].reshape(-1, 1)).to(DEVICE)   # the recorded frequency
    sw = Swing(h_start).to(DEVICE)                            # the network and beta, on the device
    def swing_loss():
        d, x = sw(tau)                                        # the network's angle and speed
        H = torch.exp(sw.beta)                                # the inertia being identified
        r1 = grad(d, tau) / (hi - lo) - x                     # the first equation: d(delta)/dt = omega - omega_s
        r2 = 2 * H / WS * grad(x, tau) / (hi - lo) - (rec["Pm"] - pb.plant_power(d, rec) - rec["D"] * x / WS)   # the second, with H on the left
        return r1.pow(2).mean() + r2.pow(2).mean() + 1000.0 * (50.0 + x / (2 * np.pi) - f_obs).pow(2).mean()   # the residuals, and the misfit
    # Adam first, then L-BFGS: the Part 2 training recipe
    train_two_stage(sw, swing_loss, adam_steps=1000, lbfgs_steps=300, lr=3e-3, report_every=0)
    with torch.no_grad():  # no gradient tracking: evaluation only
        rms = float((50.0 + sw(tau)[1] / (2 * np.pi) - f_obs).pow(2).mean().sqrt())
    return float(torch.exp(sw.beta.detach())), rms           # the inertia found (detached: a number, not a graph), and the misfit

inertia = {}
t0 = time.perf_counter()
for name, lo, hi in (("after the fault", 0.6, 2.6), ("before the fault", 0.0, 0.45)):
    keep = (rec["t"] >= lo - 1e-9) & (rec["t"] <= hi + 1e-9)
    for h_start in (8.0, 2.0):
        inertia[name, h_start] = pb.shoot_inertia(rec["t"][keep], rec["f"][keep], h_start, rec) + identify(lo, hi, h_start)
        hs, rs, hp, rp = inertia[name, h_start]
        print(f"  {name:17s} from {h_start:.0f} s: least squares H = {hs:5.2f} s (misfit {rs:.3f} Hz), physics-informed H = {hp:5.2f} s (misfit {rp:.3f} Hz)")
print(f"  the record was made with H = {rec['H_true']:.2f} s; all eight fits took {time.perf_counter() - t0:.0f} s")

**What you should see.** After the fault, **the physics-informed fit returns
3.89 s from both starts**, 3 % under the 4.00 s the record was made with, and
fits the record to its noise, 0.002 Hz. The 3 % is the model's error: the
grid was taken as fixed. **The classical fit stays near where it was
started**, 7.41 s and 1.83 s, with a misfit of 0.45 Hz. A swing integrated
with the wrong inertia rings at the wrong pitch, drifts out of step with the
record, and small changes of $H$ no longer improve it; started close to the
answer it finds 3.90 s.

Before the fault nothing moves, and **the four fits return 50.00, 1.94, 8.35
and 2.10 s, each with a perfect misfit**: near their own start, or at the
bound the fit was given. The record does not contain the inertia there, and
nothing in the fit says so (L12.1's *Inertia and Damping, Recovered
From a Disturbance*).

---

## 7 · What the notebook says

* **Six readings do not fix eleven numbers.** Weighted least squares on the
  thin set returned a state, and it was wrong at the buses with no meter.
  What rescued it was not a better algorithm but more information: a forecast
  of the injections, entered as a poor meter.
* **The network matched least squares without a single true state.** Its
  loss was the power flow equations applied to its own output. That is what
  physics-informed means here.
* **What the network bought was time.** The same accuracy from the same
  readings, in a fraction of the time once trained - which matters when the
  estimate is wanted for a thousand possible states of the grid, and not when
  it is wanted for one.
* **Both rest on the model of the grid.** A line out that nobody reported made
  both wrong by ten times. The network's own check, how well its state
  explains the readings, showed it. With no truth at hand, that check is what
  there is.
* **The inverse problem needed one more trainable number, and the equation
  written with it on the left.** Written as $\dot\omega = \ldots/H$, a very
  large $H$ makes the residual small whatever the swing does.
* **A fit always returns a number.** From the quiet window it returned a number
  near the one it was started with, or the bound it was allowed. Start a fit from two places before believing it.
* **The grid is not Denmark's.** Its impedances are textbook values and the
  accuracy of a forecast is assumed.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex12.1_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L12.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The thin set has six readings for eleven unknowns. What did weighted least squares return at the buses with no meter, and what was it that made the estimate usable again? (-> L12.1 Q2)": """
""",
    "The network was never shown a true state. What was its loss, and what would it have learned if it had been given true states instead? With a line out, how did you see that its estimate was not to be trusted? (-> L12.1 Q1)": """
""",
    "What does the swing equation add to the power flow equations, and what in it sets how fast the plant swings? (-> L12.1 Q3)": """
""",
    "The two inertia fits were started from 8 s and from 2 s. Say what each returned after the fault and before it, and how you would know from the fits alone which window to believe. (-> L12.1 Q4)": """
""",
    "To conclude: which of the numbers in your tables would you show an operator, which would you not, and why? (-> L12.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| forecast set, 300 operating points | RMSE, magnitude | RMSE, angle | worst bus, magnitude | one estimate | training |", "|---|---|---|---|---|---|",
            f"| weighted least squares | {rmse_wls[0]:.4f} p.u. | {rmse_wls[1]:.3f} deg | {wls['forecast'][0]:.4f} p.u. | {1000 * wls['forecast'][2]:.2f} ms | - |",
            f"| network, trained with the power flow | {rmse_net[0]:.4f} p.u. | {rmse_net[1]:.3f} deg | {net_score[0]:.4f} p.u. | {1000 * t_one:.2f} ms | {train_time:.0f} s |",
            "", f"Weighted least squares: full set {wls['full'][0]:.4f} p.u., thin set {wls['thin'][0]:.4f} p.u.",
            "", f"Line 1-2 out: least squares told {worst(V_a):.4f} p.u., not told {worst(V_b):.4f} p.u., network {worst(V_c):.4f} p.u.; "
            f"the network's own check (median) {np.median(J_net):.2f} -> {np.median(J_out):.2f}",
            "", "| window | started at | least squares H | physics-informed H |", "|---|---|---|---|"]
    rows += [f"| {name} | {h0:.0f} s | {v[0]:.2f} s | {v[2]:.2f} s |" for (name, h0), v in inertia.items()]
    rows += ["", f"The record was made with H = {rec['H_true']:.2f} s. The grid's impedances are textbook values, not measured."]
    out = ["# Ex_12.1 — The State of a Grid", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex12.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex12.1_report.md into Ex12.1_report.pdf, with any figure
# saved as Ex12.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex12.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex12.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex12.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex12.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini projects

Ex_12's two mini projects grow out of Ex_12.2's contingency screen; they are listed
here too, as the block's mini projects.

Each mini project takes this set's problem further - a harder version of it, or its
inverse - with no exact solution. Each student chooses **one** mini project from the
Part 2 sets and solves it individually in one month; the dates and the hand-in are
given in L13. Each mini project is **one Colab notebook in five parts** under course
policy C14: A the problem, B the data and the physics and C the reference model and
the ground truth are given and solved; D the physics-informed network and E the
report are yours. The goal is the reference model's accuracy with an online answer
at least ten times faster (for an inverse problem, ten times faster than a new
least-squares fit); each project's README states the problem, what changed, the
ground truth, a worked example and the requirements.

| mini project | new against the exercise | required | notebook |
|---|---|---|---|
| [**MP12A · Stability Estimation after Contingencies**](https://github.com/RemusTeodorescu/DL-for-Engineers-Public-Course/tree/main/miniprojects/MP12A) | the IEEE 39-bus grid, 35 contingencies, 0 to 60 % of the generation from converters with no inertia | critical clearing time within 5 ms on average, no insecure case missed, at most 5 % false alarms | [open in Colab](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP12A/MP12A_stability_estimation.ipynb) |
| [**MP12B · Stability Prediction after a Contingency**](https://github.com/RemusTeodorescu/DL-for-Engineers-Public-Course/tree/main/miniprojects/MP12B) | after the fault: 200 ms of noisy PMU-like records predict the rotor angles to 3 s and the verdict | at most 1 % of losses of synchronism missed and 5 % false alarms, angles within 5° (median) | [open in Colab](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP12B/MP12B_stability_prediction.ipynb) |